<a href="https://colab.research.google.com/github/irissouza7/Mestrado-IDP_Publico/blob/main/Consolidacao_Servicos_Arquivos_SINAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importar dados dos arquivos

In [18]:
import pandas as pd
import os
from collections import Counter

def clean_column_names(df):
    # Get initial cleaned names, allowing temporary duplicates
    cleaned_names_temp = []
    for col in df.columns:
        cleaned_col = str(col).replace('\n', '').strip().lower().replace(' ', '_')
        cleaned_names_temp.append(cleaned_col)

    # Resolve duplicates by appending suffixes
    final_unique_cols = []
    counts = Counter(cleaned_names_temp)
    seen_counts = Counter() # To keep track of how many times a cleaned name has been added
    for col_name in cleaned_names_temp:
        seen_counts[col_name] += 1
        if counts[col_name] > 1:
            # If it's a duplicate, append a unique suffix based on its occurrence.
            # Example: 'col' -> 'col_1', 'col' -> 'col_2'
            final_unique_cols.append(f"{col_name}_{seen_counts[col_name]}")
        else:
            final_unique_cols.append(col_name)

    df.columns = final_unique_cols
    return df

In [19]:

# Definição dos arquivos
#ARQUIVO_A = r"SINAPI_Referência_2025_01.xlsx"
ARQUIVO_B = r"SINAPI_referencia_servicos_consolidada_jan25_jul26.xlsx"


In [20]:
from google.colab import files

# Clique no botão 'Choose Files' e selecione todos os arquivos Excel da lista.
print("Por favor, carregue os arquivos SINAPI_Referência_YYYY_MM.xlsx:")
files.upload()

Por favor, carregue os arquivos SINAPI_Referência_YYYY_MM.xlsx:


Saving SINAPI_Referência_2025_01.xlsx to SINAPI_Referência_2025_01 (1).xlsx
Saving SINAPI_Referência_2025_02.xlsx to SINAPI_Referência_2025_02.xlsx
Saving SINAPI_Referência_2025_03.xlsx to SINAPI_Referência_2025_03.xlsx
Saving SINAPI_Referência_2025_04.xlsx to SINAPI_Referência_2025_04.xlsx
Saving SINAPI_Referência_2025_05.xlsx to SINAPI_Referência_2025_05.xlsx
Saving SINAPI_Referência_2025_06.xlsx to SINAPI_Referência_2025_06.xlsx
Saving SINAPI_Referência_2025_07.xlsx to SINAPI_Referência_2025_07.xlsx
Saving SINAPI_Referência_2025_08.xlsx to SINAPI_Referência_2025_08.xlsx
Saving SINAPI_Referência_2025_09.xlsx to SINAPI_Referência_2025_09.xlsx
Saving SINAPI_Referência_2025_10.xlsx to SINAPI_Referência_2025_10.xlsx
Saving SINAPI_Referência_2025_11.xlsx to SINAPI_Referência_2025_11.xlsx
Saving SINAPI_Referência_2025_12.xlsx to SINAPI_Referência_2025_12.xlsx
Saving SINAPI_Referência_2026_01.xlsx to SINAPI_Referência_2026_01.xlsx
Saving SINAPI_Referência_2026_02.xlsx to SINAPI_Referência_2

In [21]:
# Lista de arquivos SINAPI a serem processados
LISTA_ARQUIVOS_SINAPI = [
    "SINAPI_Referência_2025_01.xlsx",
    "SINAPI_Referência_2025_02.xlsx",
    "SINAPI_Referência_2025_03.xlsx",
    "SINAPI_Referência_2025_04.xlsx",
    "SINAPI_Referência_2025_05.xlsx",
    "SINAPI_Referência_2025_06.xlsx",
    "SINAPI_Referência_2025_07.xlsx",
    "SINAPI_Referência_2025_08.xlsx",
    "SINAPI_Referência_2025_09.xlsx",
    "SINAPI_Referência_2025_10.xlsx",
    "SINAPI_Referência_2025_11.xlsx",
    "SINAPI_Referência_2025_12.xlsx",
    "SINAPI_Referência_2026_01.xlsx",
    "SINAPI_Referência_2026_02.xlsx",
    "SINAPI_Referência_2026_03.xlsx",
    "SINAPI_Referência_2026_04.xlsx",
    "SINAPI_Referência_2026_05.xlsx",
    "SINAPI_Referência_2026_06.xlsx",
    "SINAPI_Referência_2026_07.xlsx"
]

print(f"Lista de arquivos SINAPI definida: {len(LISTA_ARQUIVOS_SINAPI)} arquivos.")

Lista de arquivos SINAPI definida: 19 arquivos.


In [22]:
all_consolidated_dfs = []

sheet_names_to_process = ["CSD", "CCD", "CSE"]

for file_path in LISTA_ARQUIVOS_SINAPI:
    print(f"Processando arquivo: {file_path}")

    # 1. Extrair ano_mes do nome do arquivo atual
    filename = os.path.basename(file_path)
    filename_without_ext = filename.replace(".xlsx", "")
    parts = filename_without_ext.split("_")

    current_ano_mes = None
    if len(parts) >= 4:
        year = parts[-2]
        month_part = parts[-1]
        month = month_part.split(' ')[0]
        current_ano_mes = f"{year}_{month}"
    else:
        print(f"Warning: Formato de nome de arquivo inesperado para {file_path}. Não foi possível extrair YYYY_MM.")
        continue # Pular para o próximo arquivo se o formato for inválido

    if not current_ano_mes:
        print(f"Erro: Não foi possível determinar ano_mes para {file_path}. Pulando este arquivo.")
        continue

    # 2. Ler todas as planilhas do arquivo atual
    try:
        excel_file = pd.ExcelFile(file_path)
        print(f"  Planilhas disponíveis: {excel_file.sheet_names}")

        processed_dfs_for_file = []
        for sheet_name in sheet_names_to_process:
            if sheet_name in excel_file.sheet_names:
                df_sheet = pd.read_excel(excel_file, sheet_name=sheet_name)
                df_sheet['Relatorio_Insumos'] = sheet_name
                df_sheet['ano_mes'] = current_ano_mes # Adicionar ano_mes específico do arquivo
                processed_dfs_for_file.append(df_sheet)
            else:
                print(f"  Warning: Planilha '{sheet_name}' não encontrada em '{file_path}'.")

        if processed_dfs_for_file:
            # Concatenar as planilhas do arquivo atual
            df_current_file = pd.concat(processed_dfs_for_file, ignore_index=True)
            # Limpar nomes das colunas
            df_current_file = clean_column_names(df_current_file)
            all_consolidated_dfs.append(df_current_file)
        else:
            print(f"  Nenhuma planilha processada para o arquivo '{file_path}'.")

    except FileNotFoundError:
        print(f"Erro: Arquivo não encontrado: {file_path}. Verifique se o arquivo está no diretório correto.")
    except Exception as e:
        print(f"Erro ao processar o arquivo {file_path}: {e}")

# 3. Concatenar todos os DataFrames processados em um único DataFrame final
if all_consolidated_dfs:
    df_consolidado = pd.concat(all_consolidated_dfs, ignore_index=True)
    print("\nProcessamento de todos os arquivos concluído com sucesso!")
    print(f"DataFrame consolidado criado com {len(df_consolidado)} linhas e {len(df_consolidado.columns)} colunas.")
    display(df_consolidado.head())
else:
    df_consolidado = pd.DataFrame()
    print("Nenhum dado foi processado ou consolidado.")

Processando arquivo: SINAPI_Referência_2025_01.xlsx
  Planilhas disponíveis: ['ISD', 'ICD', 'ISE', 'CSD', 'CCD', 'CSE', 'Analítico', 'Analítico com Custo']
Processando arquivo: SINAPI_Referência_2025_02.xlsx
  Planilhas disponíveis: ['ISD', 'ICD', 'ISE', 'CSD', 'CCD', 'CSE', 'Analítico', 'Analítico com Custo']
Processando arquivo: SINAPI_Referência_2025_03.xlsx
  Planilhas disponíveis: ['ISD', 'ICD', 'ISE', 'CSD', 'CCD', 'CSE', 'Analítico', 'Analítico com Custo']
Processando arquivo: SINAPI_Referência_2025_04.xlsx
  Planilhas disponíveis: ['ISD', 'ICD', 'ISE', 'CSD', 'CCD', 'CSE', 'Analítico', 'Analítico com Custo']
Processando arquivo: SINAPI_Referência_2025_05.xlsx
  Planilhas disponíveis: ['ISD', 'ICD', 'ISE', 'CSD', 'CCD', 'CSE', 'Analítico', 'Analítico com Custo']
Processando arquivo: SINAPI_Referência_2025_06.xlsx
  Planilhas disponíveis: ['ISD', 'ICD', 'ISE', 'CSD', 'CCD', 'CSE', 'Analítico', 'Analítico com Custo']
Processando arquivo: SINAPI_Referência_2025_07.xlsx
  Planilhas 

,grupo,código_dacomposição,descrição,unidade,ac,al,am,ap,ba,ce,...,rs,sc,se,sp,to,relatorio_insumos,ano_mes,custo_(r$),unnamed:_15,unnamed:_17
0,Acessibilidade,104658,"PISO PODOTÁTIL DE ALERTA OU DIRECIONAL, DE CON...",M2,239.55,141.03,194.49,169.11,153.84,147.41,...,148.66,140.72,157.15,186.13,186.44,CSD,2025_01,NaN,NaN,NaN
1,Acessibilidade,105002,RAMPA DE ACESSIBILIDADE EM CONCRETO MOLDADO IN...,UN,979.92,676.32,869.66,872.63,775.58,735.71,...,725.19,760.52,718.11,744.78,786.97,CSD,2025_01,NaN,NaN,NaN
2,Acessibilidade,105004,RAMPA DE ACESSIBILIDADE EM CONCRETO MOLDADO IN...,M2,169.18,114.54,149.83,150.77,132.65,125.14,...,124.54,130.82,121.47,126.17,133.95,CSD,2025_01,NaN,NaN,NaN
3,Acessibilidade,105003,RAMPA DE ACESSIBILIDADE EM CONCRETO MOLDADO IN...,UN,1490.80,1072.49,1409.00,1298.02,1300.50,1209.86,...,1232.96,1216.54,1147.10,1278.12,1225.26,CSD,2025_01,NaN,NaN,NaN
4,Acessibilidade,105005,RAMPA DE ACESSIBILIDADE EM CONCRETO MOLDADO IN...,M2,263.79,187.90,249.69,229.53,229.86,212.94,...,218.55,215.27,200.90,224.93,215.09,CSD,2025_01,NaN,NaN,NaN


In [23]:
df_consolidado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 570441 entries, 0 to 570440
Data columns (total 36 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   grupo                570441 non-null  object 
 1   código_dacomposição  570441 non-null  int64  
 2   descrição            570441 non-null  object 
 3   unidade              570441 non-null  object 
 4   ac                   560472 non-null  float64
 5   al                   570441 non-null  float64
 6   am                   570441 non-null  float64
 7   ap                   570441 non-null  float64
 8   ba                   570441 non-null  float64
 9   ce                   570441 non-null  float64
 10  df                   570441 non-null  float64
 11  es                   570441 non-null  float64
 12  go                   570441 non-null  float64
 13  ma                   570441 non-null  float64
 14  mg                   570441 non-null  float64
 15  ms               

In [24]:
df_consolidado.describe()

,código_dacomposição,ac,al,am,ap,ba,ce,df,es,go,...,ro,rr,rs,sc,se,sp,to,custo_(r$),unnamed:_15,unnamed:_17
count,570441.000000,560472.000000,570441.000000,570441.000000,570441.000000,570441.000000,570441.000000,570441.000000,570441.000000,570441.000000,...,570441.000000,570441.000000,570441.000000,570441.000000,570441.000000,570441.000000,560718.000000,19692.000000,8034.000000,8035.000000
mean,97168.336508,771.483353,583.963272,706.457187,669.206786,625.572496,586.457204,629.215177,633.783772,602.249362,...,694.350784,701.999600,604.849883,634.413027,597.134091,601.595384,603.659295,644.891471,0.109737,0.165493
std,12305.825300,4995.253530,3604.032562,4579.233527,4398.658508,3748.357527,3366.369454,3780.519217,3949.646596,3697.673361,...,4338.859526,4315.432727,3511.472480,3761.525520,3781.269579,3513.022164,3638.441985,4142.017028,0.263389,0.310805
min,5089.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,92913.000000,3.940000,3.550000,3.990000,3.380000,3.960000,3.840000,3.830000,4.040000,3.720000,...,4.070000,3.860000,3.820000,3.770000,3.550000,4.120000,3.890000,3.480000,0.000000,0.000000
50%,100242.000000,33.660000,30.330000,34.080000,30.040000,33.320000,31.920000,33.970000,35.200000,31.570000,...,33.490000,32.460000,33.510000,33.740000,30.040000,35.330000,31.640000,28.265000,0.000000,0.000000
75%,103499.000000,176.012500,148.650000,166.960000,161.500000,156.720000,155.070000,158.580000,160.090000,152.240000,...,171.600000,169.220000,152.100000,157.660000,149.940000,151.570000,155.860000,153.555000,0.009975,0.112500
max,107376.000000,152455.100000,128967.100000,143999.200000,140957.140000,140702.390000,129014.320000,129031.980000,129177.690000,128995.010000,...,136057.430000,129095.610000,128999.370000,129218.790000,128926.360000,129302.700000,129003.950000,137235.130000,1.000000,1.000000


In [26]:
df_consolidado["relatorio_insumos"].unique() # valores únicos

array(['CSD', 'CCD', 'CSE'], dtype=object)

In [10]:
df_consolidado["ano_mes"].unique() # valores únicos

array(['2025_01'], dtype=object)

In [11]:
df_consolidado["relatorio_insumos"].value_counts() # frequência dos valores

,count
relatorio_insumos,
CSD,9668
CCD,9668
CSE,9668


In [29]:
df_consolidado["ano_mes"].value_counts() # frequência dos valores

,count
ano_mes,
2026_07,31632
2026_06,31362
2026_05,31239
2026_04,31134
2026_02,31080
2026_03,30852
2026_01,30765
2025_12,30423
2025_11,29907


In [30]:
#Carregar dados do Frame df_consolidado na tabela SINAPI_referencia_servicos_consolidada_jan25_jul26.xlsx disponível no google colab - ARQUIVO_B

output_file_name = ARQUIVO_B

try:
    df_consolidado.to_excel(output_file_name, index=False) # index=False para não salvar o índice do DataFrame como uma coluna
    print(f"DataFrame salvo com sucesso em '{output_file_name}'")
except Exception as e:
    print(f"Erro ao salvar o DataFrame: {e}")

DataFrame salvo com sucesso em 'SINAPI_referencia_servicos_consolidada_jan25_jul26.xlsx'


Faz o download do arquivo chamado SINAPI_referencia_consolidada_jan25_jul26.xlsx que está salvo no diretório atual do Colab.

In [31]:
from google.colab import files

files.download('SINAPI_referencia_servicos_consolidada_jan25_jul26.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Converter arquivo xlsx em csv


In [32]:
# Nome do arquivo de entrada (Excel)
arquivo_xlsx = "SINAPI_referencia_servicos_consolidada_jan25_jul26.xlsx"

# Nome do arquivo de saída (CSV)
arquivo_csv = "SINAPI_referencia_servicos_consolidada_jan25_jul26.csv"

# Ler o arquivo Excel
df = pd.read_excel(arquivo_xlsx)

# Converter e salvar como CSV
df.to_csv(arquivo_csv, index=False, sep=";")

print(f"Arquivo convertido e salvo como: {arquivo_csv}")

Arquivo convertido e salvo como: SINAPI_referencia_servicos_consolidada_jan25_jul26.csv


Fazer download do arquivo em csv

In [33]:
from google.colab import files

files.download('SINAPI_referencia_servicos_consolidada_jan25_jul26.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



Esse código verifica se um arquivo existe e, se existir, o apaga. Caso contrário, informa que não foi encontrado.



In [16]:
import os

if os.path.exists(ARQUIVO_B):
    os.remove(ARQUIVO_B)
    print(f"O arquivo {ARQUIVO_B} foi excluído com sucesso.")
else:    print(f"O arquivo {ARQUIVO_B} não foi encontrado.")

O arquivo SINAPI_referencia_servicos_consolidada_jan25_jul26.xlsx foi excluído com sucesso.


In [17]:
# Limpar os principais DataFrames e a lista de DataFrames
df_consolidado = []
df_novos = pd.DataFrame() # Reinicia df_novos como um DataFrame vazio
# df_final = pd.DataFrame() # df_final é criado mais adiante, então não é necessário resetar aqui explicitamente se for sempre concatenado

print("Os DataFrames 'dfs' (lista) e 'df_novos' foram resetados.")
print("Você pode agora re-executar as células para carregar e processar novos dados.")

Os DataFrames 'dfs' (lista) e 'df_novos' foram resetados.
Você pode agora re-executar as células para carregar e processar novos dados.
